In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import pandas as pd
import numpy as np

# ==========================================
# 1. PERSIAPAN DATA SEDERHANA (Simulasi)
# ==========================================
# Kita load data untuk meng-adapt layer preprocessing
file_path = '/content/drive/MyDrive/Capstone/dataset_it_careers_clean.csv'
df = pd.read_csv(file_path)

# Handling Missing Values (Wajib sebelum masuk TF)
df['all_skills'] = df['all_skills'].fillna('none').astype(str).str.replace(';', ' ')
df['tools'] = df['tools'].fillna('none').astype(str).str.replace(';', ' ')
df['databases'] = df['databases'].fillna('none').astype(str).str.replace(';', ' ')
df['years_code'] = df['years_code'].fillna(0.0).astype(float)
df['education_level'] = df['education_level'].fillna(0).astype(int)

# ==========================================
# 2. DEFINISI INPUT LAYERS (TF Functional API)
# ==========================================
# Input Numerik
input_years_code = tf.keras.Input(shape=(1,), dtype=tf.float32, name='years_code')
input_edu_level = tf.keras.Input(shape=(1,), dtype=tf.float32, name='education_level')

# Input Multi-Teks (Skills, Tools, Databases)
input_skills = tf.keras.Input(shape=(1,), dtype=tf.string, name='all_skills')
input_tools = tf.keras.Input(shape=(1,), dtype=tf.string, name='tools')
input_databases = tf.keras.Input(shape=(1,), dtype=tf.string, name='databases')

# ==========================================
# 3. MERANCANG PREPROCESSING LAYERS (MANDIRI)
# ==========================================
# A. Preprocessing Numerik
norm_years = layers.Normalization(name='norm_years')
norm_years.adapt(df['years_code'].to_numpy())

norm_edu = layers.Normalization(name='norm_edu') # Bisa juga dijadikan Categorical Embedding jika dianggap non-ordinal
norm_edu.adapt(df['education_level'].to_numpy())

# B. Preprocessing Teks (TextVectorization)
# Kita bangun vocabulary untuk masing-masing teks secara mandiri (Kriteria 5)
MAX_VOCAB = 5000
MAX_LEN = 50 # Asumsi maksimal ada 50 skill/tools per user

vectorize_skills = layers.TextVectorization(max_tokens=MAX_VOCAB, output_sequence_length=MAX_LEN, name='vec_skills')
vectorize_skills.adapt(df['all_skills'].to_numpy())

vectorize_tools = layers.TextVectorization(max_tokens=MAX_VOCAB, output_sequence_length=20, name='vec_tools')
vectorize_tools.adapt(df['tools'].to_numpy())

vectorize_databases = layers.TextVectorization(max_tokens=MAX_VOCAB, output_sequence_length=15, name='vec_databases')
vectorize_databases.adapt(df['databases'].to_numpy())

# ==========================================
# 4. MENERAPKAN INPUT KE PREPROCESSING
# ==========================================
# Mengalirkan input ke layer normalisasi
processed_years = norm_years(input_years_code)
processed_edu = norm_edu(input_edu_level)

# Mengalirkan input teks ke Vectorization lalu ke Embedding layer baru
# Embedding membutuhkan input integer dari Vectorization
embed_dim = 32

# Skills Branch
vec_skills_out = vectorize_skills(input_skills)
embed_skills = layers.Embedding(input_dim=MAX_VOCAB, output_dim=embed_dim, mask_zero=True, name='embed_skills')(vec_skills_out)
pool_skills = layers.GlobalAveragePooling1D(name='pool_skills')(embed_skills)

# Tools Branch
vec_tools_out = vectorize_tools(input_tools)
embed_tools = layers.Embedding(input_dim=MAX_VOCAB, output_dim=embed_dim, mask_zero=True, name='embed_tools')(vec_tools_out)
pool_tools = layers.GlobalAveragePooling1D(name='pool_tools')(embed_tools)

# Databases Branch
vec_databases_out = vectorize_databases(input_databases)
embed_databases = layers.Embedding(input_dim=MAX_VOCAB, output_dim=embed_dim, mask_zero=True, name='embed_databases')(vec_databases_out)
pool_databases = layers.GlobalAveragePooling1D(name='pool_databases')(embed_databases)

In [ ]:
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.losses import Loss

# ==========================================
# 1. CUSTOM LOSS FUNCTION: ASYMMETRIC FOCAL LOSS
# ==========================================
class AsymmetricFocalLoss(Loss):
    def __init__(self, gamma_pos=2.0, gamma_neg=4.0, name="asymmetric_focal_loss"):
        """
        gamma_pos: Penalti untuk kesalahan prediksi pada label positif (karir yang seharusnya cocok).
        gamma_neg: Penalti untuk kesalahan prediksi pada label negatif (karir yang tidak cocok).
        Angka gamma_neg dibuat lebih besar karena dalam multi-label, nilai 0 (negatif) jauh lebih banyak dari 1.
        """
        super().__init__(name=name)
        self.gamma_pos = gamma_pos
        self.gamma_neg = gamma_neg

    def call(self, y_true, y_pred):
        # Membatasi nilai prediksi agar tidak bernilai 0 mutlak atau 1 mutlak (mencegah log(0) error)
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
        y_true = tf.cast(y_true, tf.float32)

        # Menghitung Binary Cross Entropy (BCE) dasar
        bce = -y_true * tf.math.log(y_pred) - (1.0 - y_true) * tf.math.log(1.0 - y_pred)

        # Menghitung bobot Focal Loss (Asymmetric)
        weight_pos = tf.pow(1.0 - y_pred, self.gamma_pos)
        weight_neg = tf.pow(y_pred, self.gamma_neg)

        # Menggabungkan bobot dengan label asli
        focal_weight = y_true * weight_pos + (1.0 - y_true) * weight_neg

        # Loss akhir
        loss = focal_weight * bce

        # Merata-ratakan loss untuk seluruh label dan batch
        return tf.reduce_mean(tf.reduce_sum(loss, axis=-1))

# ==========================================
# 2. CUSTOM CALLBACK: TARGET MONITOR & EARLY STOPPER
# ==========================================
class CustomTargetMonitor(Callback):
    def __init__(self, target_val_loss=0.05):
        super().__init__()
        self.target_val_loss = target_val_loss

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val_loss = logs.get('val_loss')
        train_loss = logs.get('loss')

        # Mencetak pesan kustom ke konsol
        print(f"\n[Custom Callback] Epoch {epoch + 1} Selesai | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        # Logika penghentian otomatis berbasis target
        if val_loss is not None and val_loss < self.target_val_loss:
            print(f"\n[!] Target Val Loss terpenuhi ({val_loss:.4f} < {self.target_val_loss}).")
            print("[!] Menghentikan training untuk mencegah model menghafal data (overfitting)...")
            self.model.stop_training = True

In [ ]:
# ==========================================
# 5. MENGGABUNGKAN SEMUA CABANG (CONCATENATE)
# ==========================================
# Menggabungkan representasi numerik dan embedding teks menjadi satu vektor panjang
merged_layer = layers.Concatenate(name='concat_all')([
    processed_years,
    processed_edu,
    pool_skills,
    pool_tools,
    pool_databases
])

# ==========================================
# 6. DEEP LEARNING LAYERS (FNN)
# ==========================================
# Layer pertama untuk mencari pola dari data yang digabungkan
x = layers.Dense(128, activation='relu', name='dense_1')(merged_layer)
x = layers.BatchNormalization(name='batch_norm_1')(x)
x = layers.Dropout(0.3, name='dropout_1')(x) # Mencegah overfitting

# Layer kedua untuk memperhalus pola
x = layers.Dense(64, activation='relu', name='dense_2')(x)
x = layers.BatchNormalization(name='batch_norm_2')(x)
x = layers.Dropout(0.2, name='dropout_2')(x)

# ==========================================
# 7. OUTPUT LAYER (MULTI-LABEL)
# ==========================================
# Berdasarkan eksplorasi dataset sebelumnya, terdapat 21 karir unik.
NUM_CAREERS = 21
# Menggunakan 'sigmoid' karena ini Multi-Label (tiap karir diprediksi 0.0 - 1.0 secara independen)
output_layer = layers.Dense(NUM_CAREERS, activation='sigmoid', name='career_output')(x)

# ==========================================
# 8. MEMBANGUN DAN MENGKOMPILASI MODEL
# ==========================================
# Mendefinisikan input awal dan output akhir
model = Model(
    inputs=[input_years_code, input_edu_level, input_skills, input_tools, input_databases],
    outputs=output_layer,
    name="Career_RecSys_MultiBranch"
)

# Kompilasi dengan Custom Loss yang dibuat di Langkah 2
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=AsymmetricFocalLoss(gamma_pos=2.0, gamma_neg=4.0), # Menggunakan Custom Loss
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.AUC(multi_label=True, name='auc_multi') # Metric tambahan yang relevan untuk multi-label
    ]
)

# Tampilkan ringkasan arsitektur (sangat berguna untuk laporan Capstone)
model.summary()

# (Opsional) Jika Anda ingin memvisualisasikan arsitektur menjadi gambar
# tf.keras.utils.plot_model(model, to_file='model_architecture.png', show_shapes=True, show_layer_names=True)

Model: "Career_RecSys_MultiBranch"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ all_skills          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tools (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ databases           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vec_skills          │ (None, 50)        │          0 │ all_skills[0][0]  │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vec_tools           │ (None, 20)        │          0 │ tools[0][0]       │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vec_databases       │ (None, 15)        │          0 │ databases[0][0]   │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ years_code          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ education_level     │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed_skills        │ (None, 50, 32)    │    160,000 │ vec_skills[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 50)        │          0 │ vec_skills[0][0]  │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed_tools         │ (None, 20, 32)    │    160,000 │ vec_tools[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 20)        │          0 │ vec_tools[0][0]   │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed_databases     │ (None, 15, 32)    │    160,000 │ vec_databases[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 15)        │          0 │ vec_databases[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ norm_years          │ (None, 1)         │     50,077 │ years_code[0][0]  │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ norm_edu            │ (None, 1)         │     50,077 │ education_level[… │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_skills         │ (None, 32)        │          0 │ embed_skills[0][… │
│ (GlobalAveragePool… │                   │            │ not_equal[0][0] 

 Total params: 603,215 (2.30 MB)

 Trainable params: 502,677 (1.92 MB)

 Non-trainable params: 100,538 (392.73 KB)

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.losses import Loss
from sklearn.model_selection import train_test_split

# 1. BERSIHKAN MEMORI GRAFIK (Mencegah error dimensi bertumpuk)
tf.keras.backend.clear_session()

# ==========================================
# 2. DEFINISI CUSTOM COMPONENTS
# ==========================================
class AsymmetricFocalLoss(Loss):
    def __init__(self, gamma_pos=2.0, gamma_neg=4.0, name="asymmetric_focal_loss"):
        super().__init__(name=name)
        self.gamma_pos = gamma_pos
        self.gamma_neg = gamma_neg

    def call(self, y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
        y_true = tf.cast(y_true, tf.float32)
        bce = -y_true * tf.math.log(y_pred) - (1.0 - y_true) * tf.math.log(1.0 - y_pred)
        weight_pos = tf.pow(1.0 - y_pred, self.gamma_pos)
        weight_neg = tf.pow(y_pred, self.gamma_neg)
        focal_weight = y_true * weight_pos + (1.0 - y_true) * weight_neg
        return tf.reduce_mean(tf.reduce_sum(focal_weight * bce, axis=-1))

class CustomTargetMonitor(Callback):
    def __init__(self, target_val_loss=0.05):
        super().__init__()
        self.target_val_loss = target_val_loss

    def on_epoch_end(self, epoch, logs=None):
        val_loss = logs.get('val_loss')
        if val_loss is not None and val_loss < self.target_val_loss:
            print(f"\n[!] Target Val Loss tercapai ({val_loss:.4f}). Menghentikan training agar tidak overfitting!")
            self.model.stop_training = True

# ==========================================
# 3. LOAD & PREPROCESS DATASET
# ==========================================
file_path = '/content/drive/MyDrive/Capstone/dataset_it_careers_clean.csv'
df = pd.read_csv(file_path)

df['all_skills'] = df['all_skills'].fillna('none').astype(str).str.replace(';', ' ')
df['tools'] = df['tools'].fillna('none').astype(str).str.replace(';', ' ')
df['databases'] = df['databases'].fillna('none').astype(str).str.replace(';', ' ')
df['years_code'] = df['years_code'].fillna(0.0).astype(float)
df['education_level'] = df['education_level'].fillna(0).astype(int)

y = pd.get_dummies(df['career_label']).astype(float).values
NUM_CAREERS = y.shape[1]

# ==========================================
# 4. ARSITEKTUR MODEL (DENGAN FIX SQUEEZE)
# ==========================================
# Inputs
input_years = tf.keras.Input(shape=(1,), dtype=tf.float32, name='years_code')
input_edu = tf.keras.Input(shape=(1,), dtype=tf.float32, name='education_level')
input_skills = tf.keras.Input(shape=(1,), dtype=tf.string, name='all_skills')
input_tools = tf.keras.Input(shape=(1,), dtype=tf.string, name='tools')
input_databases = tf.keras.Input(shape=(1,), dtype=tf.string, name='databases')

# Preprocessing Numerik
norm_years = layers.Normalization()
norm_years.adapt(df['years_code'].to_numpy().reshape(-1, 1))
processed_years = norm_years(input_years)

norm_edu = layers.Normalization()
norm_edu.adapt(df['education_level'].to_numpy().reshape(-1, 1))
processed_edu = norm_edu(input_edu)

# Preprocessing Teks (FIX: Menggunakan Lambda Squeeze untuk membuang dimensi ekstra)
squeeze_skills = layers.Lambda(lambda x: tf.squeeze(x, axis=-1))(input_skills)
squeeze_tools = layers.Lambda(lambda x: tf.squeeze(x, axis=-1))(input_tools)
squeeze_databases = layers.Lambda(lambda x: tf.squeeze(x, axis=-1))(input_databases)

MAX_VOCAB = 5000
embed_dim = 32

vec_skills = layers.TextVectorization(max_tokens=MAX_VOCAB, output_sequence_length=50)
vec_skills.adapt(df['all_skills'].to_numpy())
emb_skills = layers.Embedding(MAX_VOCAB, embed_dim, mask_zero=True)(vec_skills(squeeze_skills))
pool_skills = layers.GlobalAveragePooling1D()(emb_skills)

vec_tools = layers.TextVectorization(max_tokens=MAX_VOCAB, output_sequence_length=20)
vec_tools.adapt(df['tools'].to_numpy())
emb_tools = layers.Embedding(MAX_VOCAB, embed_dim, mask_zero=True)(vec_tools(squeeze_tools))
pool_tools = layers.GlobalAveragePooling1D()(emb_tools)

vec_databases = layers.TextVectorization(max_tokens=MAX_VOCAB, output_sequence_length=15)
vec_databases.adapt(df['databases'].to_numpy())
emb_databases = layers.Embedding(MAX_VOCAB, embed_dim, mask_zero=True)(vec_databases(squeeze_databases))
pool_databases = layers.GlobalAveragePooling1D()(emb_databases)

# Penggabungan & FNN
merged = layers.Concatenate()([processed_years, processed_edu, pool_skills, pool_tools, pool_databases])
x = layers.Dense(128, activation='relu')(merged)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)
output_layer = layers.Dense(NUM_CAREERS, activation='sigmoid', name='output')(x)

model = Model(inputs=[input_years, input_edu, input_skills, input_tools, input_databases], outputs=output_layer)
model.compile(optimizer='adam', loss=AsymmetricFocalLoss(), metrics=['accuracy'])

# ==========================================
# 5. DATA PIPELINE & TRAINING
# ==========================================
X = {
    'years_code': df['years_code'].to_numpy().reshape(-1, 1),
    'education_level': df['education_level'].to_numpy().reshape(-1, 1),
    'all_skills': df['all_skills'].to_numpy().reshape(-1, 1),
    'tools': df['tools'].to_numpy().reshape(-1, 1),
    'databases': df['databases'].to_numpy().reshape(-1, 1)
}

train_idx, val_idx = train_test_split(np.arange(len(df)), test_size=0.2, random_state=42)

train_ds = tf.data.Dataset.from_tensor_slices(({k: v[train_idx] for k, v in X.items()}, y[train_idx]))
train_ds = train_ds.shuffle(1024).batch(32).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices(({k: v[val_idx] for k, v in X.items()}, y[val_idx]))
val_ds = val_ds.batch(32).prefetch(tf.data.AUTOTUNE)

print("\n[INFO] Memulai Proses Training...")
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[CustomTargetMonitor(0.05), tf.keras.callbacks.EarlyStopping(patience=3)]
)

model.save('career_recsys_model.keras')
print("\n[SUCCESS] Model disimpan ke 'career_recsys_model.keras'")


[INFO] Memulai Proses Training...
Epoch 1/15
626/626 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.2753 - loss: 0.9829 - val_accuracy: 0.4804 - val_loss: 0.3652
Epoch 2/15
626/626 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.4682 - loss: 0.4028 - val_accuracy: 0.5230 - val_loss: 0.3343
Epoch 3/15
626/626 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.5111 - loss: 0.3563 - val_accuracy: 0.5369 - val_loss: 0.3200
Epoch 4/15
626/626 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.5358 - loss: 0.3333 - val_accuracy: 0.5417 - val_loss: 0.3127
Epoch 5/15
626/626 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.5419 - loss: 0.3208 - val_accuracy: 0.5453 - val_loss: 0.3085
Epoch 6/15
626/626 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.5446 - loss: 0.3140 - val_accuracy: 0.5493 - val_loss: 0.3062
Epoch 7/15
626/626 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.5506 - loss: 0.3096 - val_accuracy: 0.5515 - val_loss: 0.3039
Epoch 8/15
626/626 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accur

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, hamming_loss

# ==========================================
# 1. MELAKUKAN PREDIKSI (DIPERBAIKI)
# ==========================================
print("[INFO] Melakukan prediksi pada data validasi...")

# FIX: Menggunakan val_ds yang sudah berwujud tf.data.Dataset
# Ini secara otomatis menghindari error "dtype object"
y_pred_prob = model.predict(val_ds, verbose=0)
y_val_true = y[val_idx]

# ==========================================
# 2. PERHITUNGAN MACRO F1-SCORE & HAMMING LOSS
# ==========================================
THRESHOLD = 0.5
y_pred_bin = (y_pred_prob >= THRESHOLD).astype(int)

macro_f1 = f1_score(y_val_true, y_pred_bin, average='macro', zero_division=0)
h_loss = hamming_loss(y_val_true, y_pred_bin)

# ==========================================
# 3. PERHITUNGAN PRECISION@K (Rekomendasi Top-3)
# ==========================================
def precision_at_k(y_true, y_pred_prob, k=3):
    N = y_true.shape[0]
    precisions = []

    for i in range(N):
        top_k_idx = np.argsort(y_pred_prob[i])[-k:]
        true_labels = np.where(y_true[i] == 1)[0]
        hits = len(set(top_k_idx) & set(true_labels))
        precisions.append(hits / k)

    return np.mean(precisions)

p_at_3 = precision_at_k(y_val_true, y_pred_prob, k=3)

# ==========================================
# 4. MENAMPILKAN HASIL EVALUASI
# ==========================================
print("\n" + "="*45)
print("HASIL EVALUASI DEEP LEARNING (VALIDATION SET)")
print("="*45)
print(f"1. Macro F1-Score : {macro_f1:.4f} (Semakin mendekati 1.0 semakin baik)")
print(f"2. Hamming Loss   : {h_loss:.4f} (Semakin mendekati 0.0 semakin baik)")
print(f"3. Precision@3    : {p_at_3:.4f} (Semakin mendekati 1.0 semakin baik)")
print("="*45)

[INFO] Melakukan prediksi pada data validasi...

HASIL EVALUASI DEEP LEARNING (VALIDATION SET)
1. Macro F1-Score : 0.1715 (Semakin mendekati 1.0 semakin baik)
2. Hamming Loss   : 0.0450 (Semakin mendekati 0.0 semakin baik)
3. Precision@3    : 0.2858 (Semakin mendekati 1.0 semakin baik)
